# Task 1 - Model Implementation

The below code loads names from a text file, builds a character vocabulary with start/end tokens, and generate padded batches for model training.



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("TrainingNames.txt") as f:
    names = [line.strip().lower() for line in f]

chars = sorted(list(set("".join(names)))) + ['<s>', '<e>']
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(chars)

def encode(name):
    return [stoi['<s>']] + [stoi[c] for c in name] + [stoi['<e>']]

def decode(seq):
    return "".join([itos[i] for i in seq if i not in (stoi['<s>'], stoi['<e>'])])

encoded_names = [encode(n) for n in names]

def get_batch(batch_size=32):
    batch = random.sample(encoded_names, batch_size)
    max_len = max(len(x) for x in batch)
    X, Y = [], []
    for seq in batch:
        x = seq[:-1] + [0]*(max_len-len(seq))
        y = seq[1:] + [0]*(max_len-len(seq))
        X.append(x)
        Y.append(y)
    return torch.tensor(X).to(device), torch.tensor(Y).to(device)

The `VanillaRNN` class:
*   **`__init__`**: Initializes an `nn.Embedding` and `nn.Linear` layers (`W_xh`, `W_hh`, `W_hy`).
*   **`forward`**: Loops through the input. Updates hidden state `h = tanh(W_xh(embedded_input) + W_hh(h))`. Outputs `y_t = W_hy(h)`.

In [2]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)

        self.W_xh = nn.Linear(embed_size, hidden_size)
        self.W_hh = nn.Linear(hidden_size, hidden_size)
        self.W_hy = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        x = self.embed(x)
        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        for t in range(seq_len):
            x_t = x[:, t, :]
            h = torch.tanh(self.W_xh(x_t) + self.W_hh(h))
            y_t = self.W_hy(h)
            outputs.append(y_t.unsqueeze(1))

        return torch.cat(outputs, dim=1)

The `BLSTM` class:
*   **`__init__`**: Initializes `nn.Embedding` and numerous `nn.Linear` layers for forward/backward LSTM gates and a final output layer.
*   **`lstm_cell`**: Implements the core LSTM cell logic (input, forget, cell, output gates, state updates).
*   **`forward`**: Processes input with both a forward (left-to-right) and backward (right-to-left) LSTM. It then concatenates the hidden states from both directions at each time step, passing them through a final linear layer for output logits.

In [3]:
import torch
import torch.nn as nn

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)

        # forward
        self.Wi_f = nn.Linear(embed_size, hidden_size)
        self.Wf_f = nn.Linear(embed_size, hidden_size)
        self.Wg_f = nn.Linear(embed_size, hidden_size)
        self.Wo_f = nn.Linear(embed_size, hidden_size)

        self.Ui_f = nn.Linear(hidden_size, hidden_size)
        self.Uf_f = nn.Linear(hidden_size, hidden_size)
        self.Ug_f = nn.Linear(hidden_size, hidden_size)
        self.Uo_f = nn.Linear(hidden_size, hidden_size)

        # backward
        self.Wi_b = nn.Linear(embed_size, hidden_size)
        self.Wf_b = nn.Linear(embed_size, hidden_size)
        self.Wg_b = nn.Linear(embed_size, hidden_size)
        self.Wo_b = nn.Linear(embed_size, hidden_size)

        self.Ui_b = nn.Linear(hidden_size, hidden_size)
        self.Uf_b = nn.Linear(hidden_size, hidden_size)
        self.Ug_b = nn.Linear(hidden_size, hidden_size)
        self.Uo_b = nn.Linear(hidden_size, hidden_size)

        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def lstm_cell(self, x, h, c, Wi, Wf, Wg, Wo, Ui, Uf, Ug, Uo):
        i = torch.sigmoid(Wi(x) + Ui(h))
        f = torch.sigmoid(Wf(x) + Uf(h))
        g = torch.tanh(Wg(x) + Ug(h))
        o = torch.sigmoid(Wo(x) + Uo(h))

        c = f * c + i * g
        h = o * torch.tanh(c)

        return h, c

    def forward(self, x):
        batch_size, seq_len = x.size()

        x = self.embed(x)

        # forward direction
        h_f = torch.zeros(batch_size, self.hidden_size).to(x.device)
        c_f = torch.zeros(batch_size, self.hidden_size).to(x.device)

        forward_outputs = []

        for t in range(seq_len):
            x_t = x[:, t, :]
            h_f, c_f = self.lstm_cell(
                x_t, h_f, c_f,
                self.Wi_f, self.Wf_f, self.Wg_f, self.Wo_f,
                self.Ui_f, self.Uf_f, self.Ug_f, self.Uo_f
            )
            forward_outputs.append(h_f)

        # backward direction
        h_b = torch.zeros(batch_size, self.hidden_size).to(x.device)
        c_b = torch.zeros(batch_size, self.hidden_size).to(x.device)

        backward_outputs = []

        for t in reversed(range(seq_len)):
            x_t = x[:, t, :]
            h_b, c_b = self.lstm_cell(
                x_t, h_b, c_b,
                self.Wi_b, self.Wf_b, self.Wg_b, self.Wo_b,
                self.Ui_b, self.Uf_b, self.Ug_b, self.Uo_b
            )
            backward_outputs.insert(0, h_b)

        outputs = []

        for t in range(seq_len):
            h = torch.cat([forward_outputs[t], backward_outputs[t]], dim=1)
            outputs.append(self.fc(h).unsqueeze(1))

        return torch.cat(outputs, dim=1)

The `AttentionRNN` class:
*   **`__init__`**: Initializes an `nn.Embedding` layer, `nn.Linear` layers for input and recurrent connections (`W_xh`, `W_hh`), and a final output layer (`fc`).
*   **`forward`**: First, it processes the input sequentially to generate hidden states via a standard RNN approach. Then, it applies self-attention: it calculates attention scores between all hidden states, uses these scores to create a context vector for each time step, and finally concatenates each hidden state with its corresponding context vector before passing them through the final linear layer for output logits.

In [4]:
import torch
import torch.nn as nn

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.W_xh = nn.Linear(embed_size, hidden_size)
        self.W_hh = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        x = self.embed(x)
        h = torch.zeros(batch_size, self.hidden_size).to(x.device)
        hidden_states = []

        for t in range(seq_len):
            x_t = x[:, t, :]
            h = torch.tanh(self.W_xh(x_t) + self.W_hh(h))
            hidden_states.append(h)

        hidden_states = torch.stack(hidden_states, dim=1)

        scores = torch.bmm(hidden_states, hidden_states.transpose(1, 2))
        attn_weights = torch.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, hidden_states)

        out = torch.cat([hidden_states, context], dim=2)
        logits = self.fc(out)

        return logits

In [5]:
def train(model, epochs=5000, lr=0.001):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for ep in range(epochs):
        X, Y = get_batch()
        logits = model(X)
        loss = loss_fn(logits.view(-1, vocab_size), Y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if ep % 100 == 0:
            print(f"Epoch {ep} Loss {loss.item():.4f}")

The `generate` function:
*   Takes a trained `model` and `max_len`.
*   Initializes with `<s>` token and generates characters one by one.
*   Uses model-specific logic (`VanillaRNN`, `BLSTM`, or `AttentionRNN`) to predict the next character.
*   Stops when `<e>` is generated or `max_len` is reached.
*   Returns the decoded name.

In [6]:
def generate(model, max_len=20):
    model.eval()
    with torch.no_grad():
        x = torch.tensor([[stoi['<s>']]]).to(device)
        result = []

        h = torch.zeros(1, model.hidden_size).to(device)
        history = []
        c = torch.zeros_like(h)

        for _ in range(max_len):
            emb = model.embed(x[:, -1])

            # Vanilla RNN
            if isinstance(model, VanillaRNN):
                h = torch.tanh(model.W_xh(emb) + model.W_hh(h))
                logits = model.W_hy(h)

            # BLSTM
            elif isinstance(model, BLSTM):
                h, c = model.lstm_cell(
                    emb, h, c,
                    model.Wi_f, model.Wf_f, model.Wg_f, model.Wo_f,
                    model.Ui_f, model.Uf_f, model.Ug_f, model.Uo_f
                )
                logits = model.fc(torch.cat([h, h], dim=1))

            # Attention RNN (new version)
            elif isinstance(model, AttentionRNN):
                h = torch.tanh(model.W_xh(emb) + model.W_hh(h))
                history.append(h)

                hs = torch.stack(history, dim=1)

                scores = torch.bmm(hs, hs.transpose(1, 2))
                attn_weights = torch.softmax(scores, dim=2)
                context = torch.bmm(attn_weights, hs)

                combined = torch.cat([hs[:, -1], context[:, -1]], dim=1)
                logits = model.fc(combined)

            probs = torch.softmax(logits, dim=1)
            idx = torch.multinomial(probs, 1).item()

            if itos[idx] == '<e>':
                break

            result.append(idx)
            x = torch.cat([x, torch.tensor([[idx]]).to(device)], dim=1)

        return decode(result)

# Task 2: Quantitative Evaluation

In [7]:
def evaluate(model, num_samples=1000):
    generated = [generate(model) for _ in range(num_samples)]

    train_set = set(names)
    gen_set = set(generated)

    novelty = len([g for g in generated if g not in train_set]) / num_samples
    diversity = len(gen_set) / num_samples

    return novelty, diversity, generated[:50]

In [8]:
all_results = []

import tempfile
import os

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    os.remove(tmp.name)
    return size_mb

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

embed_size = 32
hidden_size = 32

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 32, hidden_size = 32
RNN        : 3932 (0.02 MB)
BLSTM      : 19612 (0.09 MB)
Attention  : 4828 (0.02 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.4171
Epoch 100 Loss 1.7080
Epoch 200 Loss 1.6365
Epoch 300 Loss 1.5410
Epoch 400 Loss 1.5941
Epoch 500 Loss 1.4227
Epoch 600 Loss 1.2782
Epoch 700 Loss 1.3388
Epoch 800 Loss 1.6774
Epoch 900 Loss 1.2295
Epoch 1000 Loss 1.2169
Epoch 1100 Loss 1.3577
Epoch 1200 Loss 1.2860
Epoch 1300 Loss 1.1337
Epoch 1400 Loss 1.2014
Epoch 1500 Loss 1.3116
Epoch 1600 Loss 1.1775
Epoch 1700 Loss 1.2289
Epoch 1800 Loss 1.2362
Epoch 1900 Loss 1.2305
Epoch 2000 Loss 1.0617
Epoch 2100 Loss 1.0996
Epoch 2200 Loss 1.0895
Epoch 2300 Loss 1.3020
Epoch 2400 Loss 1.1333
Epoch 2500 Loss 1.0890
Epoch 2600 Loss 1.1201
Epoch 2700 Loss 1.1837
Epoch 2800 Loss 1.1768
Epoch 2900 Loss 1.0364
Epoch 3000 Loss 1.0548
Epoch 3100 Loss 1.1218
Epoch 3200 Loss 1.0084
Epoch 

In [9]:
embed_size = 32
hidden_size = 64

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 32, hidden_size = 64
RNN        : 8988 (0.04 MB)
BLSTM      : 54684 (0.22 MB)
Attention  : 10780 (0.04 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.3734
Epoch 100 Loss 1.5958
Epoch 200 Loss 1.4577
Epoch 300 Loss 1.4151
Epoch 400 Loss 1.2006
Epoch 500 Loss 1.2702
Epoch 600 Loss 1.2072
Epoch 700 Loss 1.2166
Epoch 800 Loss 1.4016
Epoch 900 Loss 0.9345
Epoch 1000 Loss 1.1660
Epoch 1100 Loss 1.0722
Epoch 1200 Loss 1.0633
Epoch 1300 Loss 1.0846
Epoch 1400 Loss 1.0513
Epoch 1500 Loss 0.9828
Epoch 1600 Loss 1.1428
Epoch 1700 Loss 1.0697
Epoch 1800 Loss 1.0778
Epoch 1900 Loss 1.1645
Epoch 2000 Loss 0.9311
Epoch 2100 Loss 0.9979
Epoch 2200 Loss 1.0493
Epoch 2300 Loss 1.0031
Epoch 2400 Loss 0.8659
Epoch 2500 Loss 0.8779
Epoch 2600 Loss 0.8249
Epoch 2700 Loss 0.8865
Epoch 2800 Loss 0.9344
Epoch 2900 Loss 0.9089
Epoch 3000 Loss 0.8612
Epoch 3100 Loss 0.9293
Epoch 3200 Loss 0.9329
Epoch

In [10]:
embed_size = 32
hidden_size = 128

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 32, hidden_size = 128
RNN        : 25244 (0.10 MB)
BLSTM      : 173980 (0.67 MB)
Attention  : 28828 (0.11 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.3362
Epoch 100 Loss 1.3747
Epoch 200 Loss 1.3184
Epoch 300 Loss 1.3430
Epoch 400 Loss 1.2312
Epoch 500 Loss 1.0683
Epoch 600 Loss 1.1565
Epoch 700 Loss 1.1311
Epoch 800 Loss 0.9036
Epoch 900 Loss 0.9298
Epoch 1000 Loss 1.1678
Epoch 1100 Loss 0.9694
Epoch 1200 Loss 0.9157
Epoch 1300 Loss 0.9673
Epoch 1400 Loss 0.9203
Epoch 1500 Loss 0.9364
Epoch 1600 Loss 0.9302
Epoch 1700 Loss 0.8011
Epoch 1800 Loss 0.7607
Epoch 1900 Loss 0.7183
Epoch 2000 Loss 0.7753
Epoch 2100 Loss 0.7048
Epoch 2200 Loss 0.7922
Epoch 2300 Loss 0.6849
Epoch 2400 Loss 0.7791
Epoch 2500 Loss 0.8097
Epoch 2600 Loss 0.7869
Epoch 2700 Loss 0.8239
Epoch 2800 Loss 0.5919
Epoch 2900 Loss 0.7405
Epoch 3000 Loss 0.6481
Epoch 3100 Loss 0.6545
Epoch 3200 Loss 0.6751
Ep

In [11]:
embed_size = 64
hidden_size = 32

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 64, hidden_size = 32
RNN        : 5852 (0.03 MB)
BLSTM      : 28700 (0.12 MB)
Attention  : 6748 (0.03 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.3966
Epoch 100 Loss 2.1587
Epoch 200 Loss 1.6356
Epoch 300 Loss 1.5560
Epoch 400 Loss 1.3877
Epoch 500 Loss 1.2304
Epoch 600 Loss 1.3847
Epoch 700 Loss 1.2261
Epoch 800 Loss 1.3696
Epoch 900 Loss 1.1914
Epoch 1000 Loss 1.2132
Epoch 1100 Loss 1.2196
Epoch 1200 Loss 1.2029
Epoch 1300 Loss 1.2004
Epoch 1400 Loss 1.2872
Epoch 1500 Loss 1.0032
Epoch 1600 Loss 1.2970
Epoch 1700 Loss 1.2214
Epoch 1800 Loss 1.1362
Epoch 1900 Loss 1.1974
Epoch 2000 Loss 1.0689
Epoch 2100 Loss 1.2037
Epoch 2200 Loss 1.2644
Epoch 2300 Loss 1.2099
Epoch 2400 Loss 1.0903
Epoch 2500 Loss 1.0875
Epoch 2600 Loss 1.0301
Epoch 2700 Loss 1.1762
Epoch 2800 Loss 1.2672
Epoch 2900 Loss 1.3239
Epoch 3000 Loss 1.0158
Epoch 3100 Loss 1.0704
Epoch 3200 Loss 1.2718
Epoch 

In [12]:
embed_size = 64
hidden_size = 64

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 64, hidden_size = 64
RNN        : 11932 (0.05 MB)
BLSTM      : 71964 (0.29 MB)
Attention  : 13724 (0.06 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.3343
Epoch 100 Loss 1.5000
Epoch 200 Loss 1.3514
Epoch 300 Loss 1.2302
Epoch 400 Loss 1.0679
Epoch 500 Loss 1.3863
Epoch 600 Loss 1.2410
Epoch 700 Loss 1.1623
Epoch 800 Loss 1.0096
Epoch 900 Loss 1.1133
Epoch 1000 Loss 1.1878
Epoch 1100 Loss 1.2253
Epoch 1200 Loss 1.0519
Epoch 1300 Loss 1.2090
Epoch 1400 Loss 1.0556
Epoch 1500 Loss 1.0725
Epoch 1600 Loss 1.0091
Epoch 1700 Loss 0.9955
Epoch 1800 Loss 0.9348
Epoch 1900 Loss 0.8242
Epoch 2000 Loss 0.8396
Epoch 2100 Loss 1.2319
Epoch 2200 Loss 1.1446
Epoch 2300 Loss 1.0189
Epoch 2400 Loss 1.0637
Epoch 2500 Loss 1.0405
Epoch 2600 Loss 0.8819
Epoch 2700 Loss 0.8734
Epoch 2800 Loss 0.8049
Epoch 2900 Loss 0.8951
Epoch 3000 Loss 0.7847
Epoch 3100 Loss 0.7747
Epoch 3200 Loss 0.8495
Epoc

In [13]:
embed_size = 64
hidden_size = 128

rnn_model = VanillaRNN(vocab_size, embed_size, hidden_size)
blstm_model = BLSTM(vocab_size, embed_size, hidden_size)
attn_model = AttentionRNN(vocab_size, embed_size, hidden_size)

print("\n================ MODEL INITIATION & PARAMETER COUNT ================")
print(f"embed_size = {embed_size}, hidden_size = {hidden_size}")
print(f"RNN        : {count_params(rnn_model)} ({get_model_size_mb(rnn_model):.2f} MB)")
print(f"BLSTM      : {count_params(blstm_model)} ({get_model_size_mb(blstm_model):.2f} MB)")
print(f"Attention  : {count_params(attn_model)} ({get_model_size_mb(attn_model):.2f} MB)")

print("\n================ TRAINING =======================")
print("\n[RNN]")
train(rnn_model)

print("\n[BLSTM]")
train(blstm_model)

print("\n[Attention RNN]")
train(attn_model)

print("\n================ EVALUATION =====================")

rnn_nov, rnn_div, rnn_samples = evaluate(rnn_model)
print(f"\n--- RNN ---")
print(f"Novelty   : {rnn_nov:.4f}")
print(f"Diversity : {rnn_div:.4f}")
print(f"Samples   : {rnn_samples}")

blstm_nov, blstm_div, blstm_samples = evaluate(blstm_model)
print(f"\n--- BLSTM ---")
print(f"Novelty   : {blstm_nov:.4f}")
print(f"Diversity : {blstm_div:.4f}")
print(f"Samples   : {blstm_samples}")

attn_nov, attn_div, attn_samples = evaluate(attn_model)
print(f"\n--- ATTN ---")
print(f"Novelty   : {attn_nov:.4f}")
print(f"Diversity : {attn_div:.4f}")
print(f"Samples   : {attn_samples}")

all_results.append({
    "config": f"E{embed_size}_H{hidden_size}",
    "RNN": (count_params(rnn_model), get_model_size_mb(rnn_model), rnn_nov, rnn_div),
    "BLSTM": (count_params(blstm_model), get_model_size_mb(blstm_model), blstm_nov, blstm_div),
    "ATTN": (count_params(attn_model), get_model_size_mb(attn_model), attn_nov, attn_div)
})


================ MODEL INITIATION & PARAMETER COUNT ================
embed_size = 64, hidden_size = 128
RNN        : 30236 (0.12 MB)
BLSTM      : 207644 (0.80 MB)
Attention  : 33820 (0.13 MB)

================ TRAINING =======================

[RNN]
Epoch 0 Loss 3.4419
Epoch 100 Loss 1.6217
Epoch 200 Loss 1.2795
Epoch 300 Loss 1.4085
Epoch 400 Loss 1.1162
Epoch 500 Loss 1.1366
Epoch 600 Loss 0.9247
Epoch 700 Loss 0.9427
Epoch 800 Loss 0.9251
Epoch 900 Loss 0.8667
Epoch 1000 Loss 0.9048
Epoch 1100 Loss 0.9417
Epoch 1200 Loss 0.9350
Epoch 1300 Loss 0.9316
Epoch 1400 Loss 0.8541
Epoch 1500 Loss 0.7960
Epoch 1600 Loss 0.7851
Epoch 1700 Loss 0.8995
Epoch 1800 Loss 0.7653
Epoch 1900 Loss 0.7108
Epoch 2000 Loss 0.6867
Epoch 2100 Loss 0.7054
Epoch 2200 Loss 0.7886
Epoch 2300 Loss 0.7963
Epoch 2400 Loss 0.7164
Epoch 2500 Loss 0.6467
Epoch 2600 Loss 0.7670
Epoch 2700 Loss 0.7485
Epoch 2800 Loss 0.7248
Epoch 2900 Loss 0.6969
Epoch 3000 Loss 0.6565
Epoch 3100 Loss 0.8012
Epoch 3200 Loss 0.7095
Ep

In [14]:
print("\n================ FINAL RESULTS TABLE =================")
print("Format: (Parameters, Size_MB, Novelty, Diversity)")

print(f"{'Config':<12} | {'RNN':<40} | {'BLSTM':<40} | {'Attention':<40}")
print("-"*140)

for row in all_results:
    config = row["config"]

    rnn_val = f"{row['RNN'][0]}, {row['RNN'][1]:.2f}MB, {row['RNN'][2]:.3f}, {row['RNN'][3]:.3f}"
    blstm_val = f"{row['BLSTM'][0]}, {row['BLSTM'][1]:.2f}MB, {row['BLSTM'][2]:.3f}, {row['BLSTM'][3]:.3f}"
    attn_val = f"{row['ATTN'][0]}, {row['ATTN'][1]:.2f}MB, {row['ATTN'][2]:.3f}, {row['ATTN'][3]:.3f}"

    print(f"{config:<12} | {rnn_val:<40} | {blstm_val:<40} | {attn_val:<40}")


================ FINAL RESULTS TABLE =================
Format: (Parameters, Size_MB, Novelty, Diversity)
Config       | RNN                                      | BLSTM                                    | Attention                               
--------------------------------------------------------------------------------------------------------------------------------------------
E32_H32      | 3932, 0.02MB, 0.946, 0.965               | 19612, 0.09MB, 0.999, 0.827              | 4828, 0.02MB, 0.931, 0.971              
E32_H64      | 8988, 0.04MB, 0.746, 0.895               | 54684, 0.22MB, 1.000, 0.992              | 10780, 0.04MB, 0.671, 0.869             
E32_H128     | 25244, 0.10MB, 0.341, 0.712              | 173980, 0.67MB, 0.995, 0.952             | 28828, 0.11MB, 0.242, 0.674             
E64_H32      | 5852, 0.03MB, 0.943, 0.958               | 28700, 0.12MB, 0.998, 0.862              | 6748, 0.03MB, 0.918, 0.952              
E64_H64      | 11932, 0.05MB, 0.710, 0.878 

### Quantitative Comparison of Models

BLSTM models consistently achieve the highest novelty (0.998-1.000) and diversity (0.827-0.992), but their high novelty might indicate a failure to learn realistic name patterns, leading to unrealistic sequences. In contrast, Vanilla RNN and Attention RNN models show decreasing novelty and diversity as model size increases (e.g., RNN novelty drops from 0.946 to 0.292), suggesting potential overfitting or less varied outputs. While BLSTM models generally have higher parameter counts (19k-207k) and size (0.09-0.80MB), both RNN and Attention RNN models are significantly smaller (3.9k-33.8k).